# Job Salary Prediction: EDA, Baseline, and Tuning

This notebook combines the original exploratory analysis, baseline modeling, and model tuning steps into a single Kaggle-friendly workflow.

## Kaggle Setup
- Add the CSV files (`job.csv`, `train.csv`, `test.csv`) as Kaggle input data, or upload them into the working directory.
- The helper cell below will try common Kaggle and local paths automatically.
- Model artifacts are saved into the current working directory or `/kaggle/working` when available.


In [ ]:
from pathlib import Path


def find_file(name: str) -> Path:
    candidates = [
        Path('/kaggle/input'),
        Path('/kaggle/working'),
        Path('.'),
        Path('data'),
        Path('../data'),
    ]

    for base in candidates:
        direct = base / name
        if direct.exists():
            return direct
        if base.exists():
            matches = list(base.rglob(name))
            if matches:
                return matches[0]

    raise FileNotFoundError(f'Could not find {name!r}. Add it as a Kaggle dataset/input or place it next to this notebook.')


JOB_CSV = find_file('job.csv')
TRAIN_CSV = find_file('train.csv')
TEST_CSV = find_file('test.csv')
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
BASELINE_MODEL_PATH = OUTPUT_DIR / 'baseline_salary_model.joblib'
TUNED_MODEL_PATH = OUTPUT_DIR / 'tuned_salary_model.joblib'

print('job.csv ->', JOB_CSV)
print('train.csv ->', TRAIN_CSV)
print('test.csv ->', TEST_CSV)
print('output dir ->', OUTPUT_DIR.resolve())


## 1. Exploratory Data Analysis and Feature Engineering


In [ ]:
# import necessay libraries
import pandas as pd
import numpy  as np
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data = pd.read_csv(JOB_CSV)
data.shape

In [ ]:
data.head()

In [ ]:
data.columns

In [ ]:
data.isnull().sum()

In [ ]:
data['job_title'].duplicated().sum()

In [ ]:
data.info()

In [ ]:
data['ctc'].value_counts()

In [ ]:
data['job_title'].value_counts()[:5]

In [ ]:
data['ctc'].unique()

In [ ]:
def clean_ctc(series):
    return (series
            .str.replace(r'[^0-9,]', '', regex=True)
            .str.replace(',', '', regex=False)
            .replace('', pd.NA)
           )

data['is_salary_disclosed'] = data['ctc'].ne('Competitive salary')
data['min_ctc'] = clean_ctc(data['ctc'].str.split('-').str[0])
data['max_ctc'] = clean_ctc(data['ctc'].str.split('-').str[-1])

# Optional: convert to numeric for calculations
data['min_ctc'] = pd.to_numeric(data['min_ctc'], errors='coerce')
data['max_ctc'] = pd.to_numeric(data['max_ctc'], errors='coerce')

In [ ]:
data['experience'].value_counts()

In [ ]:
data['experience'] = data['experience'].str.replace(' years', "")
data['min_exp'] = clean_ctc(data['experience'].str.split('-').str[0])
data['max_exp'] = clean_ctc(data['experience'].str.split('-').str[-1])

# Optional: convert to numeric for calculations
data['min_exp'] = pd.to_numeric(data['min_exp'], errors='coerce')
data['max_exp'] = pd.to_numeric(data['max_exp'], errors='coerce')

In [ ]:
data.info()

In [ ]:
data[data.isna().any(axis= 1)]

## "Competitive salary" means salary is undisclosed, so we keep min_ctc and max_ctc as missing values

In [ ]:
# Extract only numeric part from "1 year", "3-5", "2-10" etc.
data['min_exp'] = data['experience'].str.extract(r'(\d+)').astype(float)
data['max_exp'] = data['experience'].str.extract(r'(\d+)-?(\d+)?')[1].fillna(data['min_exp']).astype(float)

In [ ]:
data[data['ctc'] == 'Competitive salary']

In [ ]:
## keep undisclosed salaries as missing to avoid inventing values
data.loc[~data['is_salary_disclosed'], ['ctc', 'min_ctc', 'max_ctc']].head()

In [ ]:
data.isnull().sum()

In [ ]:
## convert float64 into nullable integer columns while preserving missing salaries
data['min_ctc'] = data['min_ctc'].astype('Int64')
data['max_ctc'] = data['max_ctc'].astype('Int64')
data['min_exp'] = data['min_exp'].astype('Int64')
data['max_exp'] = data['max_exp'].astype('Int64')

In [ ]:
data.describe()

In [ ]:
top_5_jobs = data['job_title'].value_counts().head(5)

plt.bar(x = top_5_jobs.index, height= top_5_jobs.values)
plt.title("Top 5 jobs")
plt.xlabel("position")
plt.ylabel("frequency of job")
plt.xticks(rotation = 90)
plt.show()

In [ ]:
# instead we can use seaborn to make it more nice
import seaborn as sns

top_5_jobs = data['job_title'].value_counts().head(5).reset_index()
top_5_jobs.columns = ['job_title', 'frequency']

sns.barplot(data=top_5_jobs, x='job_title', y='frequency', palette='viridis')
plt.title("Top 5 Jobs")
plt.xlabel("Position")
plt.ylabel("Frequency of Job")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
data['posted'].unique()

In [ ]:
data['posted'] = data["posted"].str.replace("\n\n\nBe an early applicant", "")
data['posted'].unique()

In [ ]:
listing_key = ['job_title', 'company_name', 'location']

exact_duplicate_mask = data.duplicated(keep='first')
repeated_listing_mask = data.duplicated(subset=listing_key, keep=False)
repost_candidate_mask = repeated_listing_mask & ~data.duplicated(keep=False)

data['is_noisy_duplicate'] = exact_duplicate_mask
data['is_repost_candidate'] = repost_candidate_mask

print(f"Exact duplicate rows to drop from summary stats: {data['is_noisy_duplicate'].sum()}")
print(f"Rows that look like reposted/refreshed listings: {data['is_repost_candidate'].sum()}")

In [ ]:
data.loc[data['is_noisy_duplicate'], ['job_title', 'company_name', 'location', 'ctc', 'experience', 'posted']].head(10)

In [ ]:
repost_candidates = (data.loc[data['is_repost_candidate']]
    .sort_values(listing_key + ['posted'])
    [['job_title', 'company_name', 'location', 'ctc', 'experience', 'posted']]
)

repost_candidates.head(10)

In [ ]:
analysis_data = data.loc[~data['is_noisy_duplicate']].copy()
print(f'Rows in original data: {len(data)}')
print(f'Rows used for summary stats: {len(analysis_data)}')
analysis_data.head()

In [ ]:
def convert_to_days(posted):
    if pd.isna(posted):
        return pd.NA
    posted = str(posted).strip()
    if posted in ['Just now', 'Few hours ago', 'Today'] or 'hour' in posted:
        return 0
    if 'week' in posted:
        return int(posted.split()[0]) * 7
    if 'day' in posted:
        return int(posted.split()[0])
    return pd.NA

analysis_data['posted_days'] = analysis_data['posted'].apply(convert_to_days).astype('Int64')
analysis_data['is_remote'] = analysis_data['location'].str.contains('Work from home', case=False, na=False)
analysis_data['avg_ctc'] = ((analysis_data['min_ctc'] + analysis_data['max_ctc']) / 2).round(0).astype('Int64')

analysis_data[['posted', 'posted_days', 'is_remote', 'avg_ctc']].head()

In [ ]:
analysis_data[['posted_days', 'is_remote', 'avg_ctc']].describe(include='all')

In [ ]:
top_10_jobs = analysis_data['job_title'].value_counts().head(10).reset_index()
top_10_jobs.columns = ['job_title', 'count']

plt.figure(figsize=(10, 6))
sns.barplot(data=top_10_jobs, x='count', y='job_title', hue='job_title', palette='viridis', legend=False)
plt.title('Top 10 Job Titles')
plt.xlabel('Number of Listings')
plt.ylabel('Job Title')
plt.tight_layout()
plt.show()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
top_10_locations = data['location'].value_counts().head(10).reset_index()
top_10_locations.columns = ['location', 'count']

plt.figure(figsize=(10, 6))
sns.barplot(data=top_10_locations, x='count', y='location', hue='location', palette='magma', legend=False)
plt.title('Top 10 Hiring Locations')
plt.xlabel('Number of Listings')
plt.ylabel('Location')
plt.tight_layout()
plt.show()

In [ ]:
top_10_companies = data['company_name'].value_counts().head(10).reset_index()
top_10_companies.columns = ['company_name', 'count']

plt.figure(figsize=(10, 6))
sns.barplot(data=top_10_companies, x='count', y='company_name', hue='company_name', palette='cubehelix', legend=False)
plt.title('Top 10 Hiring Companies')
plt.xlabel('Number of Listings')
plt.ylabel('Company')
plt.tight_layout()
plt.show()

## Feature engineering

In [ ]:
analysis_data['avg_ctc'].hist(bins = 50)
plt.yscale('log') # y-axis to a logarithmic scale
plt.title('Avg CTC distribution (log scale)')

In [ ]:
# correlations
# analysis_data.info()
numeric_cols = analysis_data.select_dtypes(include= 'number').columns.tolist()
sns.heatmap(analysis_data[numeric_cols].corr(), annot= True)

In [ ]:
## outlier capping(winzorization): means limiting extreme values in your data to a set boundary, instead of removing them.
for col in ['avg_ctc', 'min_ctc', 'max_ctc']:
    upper = analysis_data[col].quantile(0.95)   ##takes 95% of the data
    analysis_data[col] = analysis_data[col].clip(upper= upper)
# categorical encoding prep
top_jobs = analysis_data['job_title'].value_counts().head(20)
analysis_data['job_title_top'] = analysis_data['job_title'].apply(lambda x:x if x in top_jobs else "other")

In [ ]:
## split the data info train and test.
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(analysis_data, test_size= 0.2, random_state= 42)
train_df.to_csv(OUTPUT_DIR / 'train.csv', index=False)
test_df.to_csv(OUTPUT_DIR / 'test.csv', index=False)


## model

In [ ]:
from xgboost import XGBRegressor

In [ ]:
test = pd.read_csv(OUTPUT_DIR / 'test.csv')
train = pd.read_csv(OUTPUT_DIR / 'train.csv')


## 2. Baseline Model


# 02. Baseline Model - Job Salary Prediction


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
import joblib
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded!')

In [ ]:
# Load splits (EDA output)
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)

print(f'Train: {train.shape}, Test: {test.shape}')
print('Target stats:')
print(train['avg_ctc'].describe())

In [ ]:
# Features (EDA features + categoricals)
features = ['min_exp', 'max_exp', 'posted_days']

# Top 20 jobs/locations for encoding
top_jobs = train['job_title'].value_counts().head(20).index
train['job_top'] = pd.Categorical(train['job_title'].where(train['job_title'].isin(top_jobs), 'Other'))
test['job_top'] = pd.Categorical(test['job_title'].where(test['job_title'].isin(top_jobs), 'Other'))

top_locations = train['location'].value_counts().head(20).index
train['loc_top'] = pd.Categorical(train['location'].where(train['location'].isin(top_locations), 'Other'))
test['loc_top'] = pd.Categorical(test['location'].where(train['location'].isin(top_locations), 'Other'))

# Dummies
all_features = features + ['job_top', 'loc_top']
X_train = pd.get_dummies(train[all_features], drop_first=True)
y_train = train['avg_ctc'].fillna(train['avg_ctc'].median())

X_test = pd.get_dummies(test[all_features], drop_first=True)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)  # Align columns
y_test = test['avg_ctc'].fillna(train['avg_ctc'].median())

print(f'Features: {X_train.shape[1]}')
print(X_train.head())

In [ ]:
# XGBoost baseline
model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Metrics
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'✅ BASELINE MODEL COMPLETE!')
print(f'MAE: ₹{mae:,.0f}')
print(f'R² Score: {r2:.3f}')
print(f'Median baseline: ₹{y_test.median():,.0f}')

In [ ]:
# Feature importance
import matplotlib.pyplot as plt
importance = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
importance.head(10).plot(kind='barh', title='Top 10 Features')
plt.tight_layout()
plt.show()

# Save model
joblib.dump(model, BASELINE_MODEL_PATH)
print('✅ Model saved! Ready for deployment.')

## 3. Model Tuning


# 03. Model Tuning - Job Salary Prediction

Hyperparameter tuning for XGBoost baseline using RandomizedSearchCV (faster) or Optuna.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
import joblib
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded!')

In [ ]:
# Load splits (same as baseline)
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)

print(f'Train: {train.shape}, Test: {test.shape}')
print('Target stats:')
print(train['avg_ctc'].describe())

In [ ]:
# Features (same preprocessing)
features = ['min_exp', 'max_exp', 'posted_days']

# Top 20 encoding (reuse baseline logic)
top_jobs = train['job_title'].value_counts().head(20).index
train['job_top'] = pd.Categorical(train['job_title'].where(train['job_title'].isin(top_jobs), 'Other'))
test['job_top'] = pd.Categorical(test['job_title'].where(test['job_title'].isin(top_jobs), 'Other'))

top_locations = train['location'].value_counts().head(20).index
train['loc_top'] = pd.Categorical(train['location'].where(train['location'].isin(top_locations), 'Other'))
test['loc_top'] = pd.Categorical(test['location'].where(test['location'].isin(top_locations), 'Other'))

# Dummies
all_features = features + ['job_top', 'loc_top']
X_train = pd.get_dummies(train[all_features], drop_first=True)
y_train = train['avg_ctc'].fillna(train['avg_ctc'].median())

X_test = pd.get_dummies(test[all_features], drop_first=True)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
y_test = test['avg_ctc'].fillna(train['avg_ctc'].median())

print(f'Features: {X_train.shape[1]}')
print(X_train.head())

In [ ]:
# Baseline recall (load and eval)
baseline_model = joblib.load(BASELINE_MODEL_PATH)
y_pred_baseline = baseline_model.predict(X_test)
mae_baseline = mean_absolute_error(y_test, y_pred_baseline)
r2_baseline = r2_score(y_test, y_pred_baseline)
print(f'Baseline - MAE: ₹{mae_baseline:,.0f}, R²: {r2_baseline:.3f}')

In [ ]:
# RandomizedSearchCV tuning
model = XGBRegressor(random_state=42, n_jobs=-1)

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

# TimeSeriesSplit for temporal data (sort by posted_days?)
tscv = TimeSeriesSplit(n_splits=5)
random_search = RandomizedSearchCV(
    model, param_distributions=param_dist, n_iter=100, 
    cv=tscv, scoring='neg_mean_absolute_error', 
    random_state=42, n_jobs=-1, verbose=1
)

random_search.fit(X_train, y_train)
print('Best params:', random_search.best_params_)
print('Best CV score:', -random_search.best_score_)

In [ ]:
# Test tuned model
best_model = random_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)
mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
r2_tuned = r2_score(y_test, y_pred_tuned)

print(f'Tuned - MAE: ₹{mae_tuned:,.0f} ({((mae_baseline - mae_tuned)/mae_baseline)*100:.1f}% improvement), R²: {r2_tuned:.3f}')

# Feature importance
importance = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
importance.head(10).plot(kind='barh', title='Tuned Top 10 Features')
import matplotlib.pyplot as plt
plt.tight_layout()
plt.show()

# Save best model
joblib.dump(best_model, TUNED_MODEL_PATH)
print('✅ Tuned model saved!')

In [ ]:
import optuna

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        # ... etc
    }
    # CV score
    return score

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100)

## Next Steps
- Run tuning (may take 10-30min)
- Compare results
- Mark TODO.md step 4-5 complete
- Optional: Try Optuna for more advanced tuning
```python
import optuna

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        # ... etc
    }
    # CV score
    return score

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100)
```